# Filling With Packmol
---
This tutorial shows how to fill a universe from scratch using the packmol integration with MDMC

### Prerequisites:
If you are not aware of how packmol works, then please read the user guide [here](https://m3g.github.io/packmol/userguide.shtml).

You will need the necessary individual molecule configurations in MDMC. (See the [_Building a Universe_](https://github.com/MDMCproject/MDMCv0.2_pilot/wiki/Building-a-Universe) notebook for how to make configurations.)

Examples of packmol input can be found [here](https://m3g.github.io/packmol/examples.shtml)

This tutorial also presumes you are running this on a docker container, with an x-server avaliable for visualisation

In [ ]:
from MDMC.MD.packmol import packmol_setup, packmol_wrapper
from MDMC.gui import view

The first example I will show is a simple system of water mixed in with ethanol.

Firstly, we create the molecules needed for the system

(NB: We highly recommend adding any atom labels, dihedrals, bond angles, etc. to the molecules at this stage to ensure they are present in the final universe)

In [ ]:
from MDMC.MD import Atom, Molecule, Bond, BondAngle
H1 = Atom('H')
H2 = Atom('H', position=[0., 1.63298, 0.])
O = Atom('O', position=[0., 0.81649, 0.57736])
HOH_angle = BondAngle(H1, O, H2)
water_molecule = Molecule(atoms=[H1, H2, O], interactions=[HOH_angle], name="water")

h_1 = Atom("H", position=[-1.9237, 0.3850, 0.0000])
h_2 = Atom("H", position=[2.0985, 0.2306, 0.0000])
h_3 = Atom("H", position=[1.1184, -1.0093, 0.8869])
h_4 = Atom("H", position=[1.1184, -1.0093, -0.8869])
h_5 = Atom("H", position=[-0.0227, 1.1812, 0.8852])
h_6 = Atom("H", position=[-0.0227, 1.1812, -0.8852])
c_1 = Atom("C", position=[1.1879, -0.3829, 0.0000])
c_2 = Atom("C", position=[0.0000, 0.5526, 0.0000])
o_1 = Atom("O", position=[-1.1867, -0.2472, 0.0000])
ch_bond = Bond((h_2, c_1), (h_3, c_1), (h_4, c_1), (h_5, c_2), (h_6, c_2))
co_bond = Bond((c_2, o_1))
cc_bond = Bond((c_1, c_2))
oh_bond = Bond((o_1, h_1))
ethanol_molecule = Molecule(atoms=[h_1,h_2,h_3,h_4,h_5,h_6,c_1,c_2,o_1], interactions=[ch_bond, co_bond, cc_bond, oh_bond], name="ethanol")

Next, put this into a `PackmolSetup` object.

Currently, only single fixed molecules, or boxes, cubes and spheres of molecules are supported.

(NB: Density is currently only provided in molecules per ang^3, you may wish to provide a number of molecules (n_molecules) to the setup instead)

In [ ]:
setup = packmol_setup.PackmolSetup()
# Two identically overlapping cubes will create a mixture of water and ethanol in a 1:1 ratio of molecules
setup.add_cube(molecule=water_molecule, size=40., density=0.01)
setup.add_cube(molecule=ethanol_molecule, size=40., density=0.01)

Next step is to pass this `PackmolSetup` object to the `fill_with_packmol` function.

This should run packmol in the background and return a filled universe with the molecules specified.

In [ ]:
universe = packmol_wrapper.fill_with_packmol(setup_data=setup)

Now we can use `view` to look at the solved system

In [ ]:
from MDMC.gui import view
view(universe, "ASE")